# Add tariff categories incrementally

This notebook adds only the new `before August 22`, `after August 22`, and `Section 338` categories. It reuses the existing NAICS employment tables and updates the existing ADA/CSD CSV and GeoJSON outputs.

Set `WRITE_OUTPUTS = True` in Cell 2 only after the validation cells look correct. The original category columns are preserved.

In [25]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import geopandas as gpd

os.environ['OGR_GEOJSON_MAX_OBJ_SIZE'] = '0'

REPO = Path.cwd().parents[1]
RAW = REPO / 'analysis' / 'raw'
INTERMEDIATE = REPO / 'analysis' / 'intermediate'
NOTEBOOK_OUTPUTS = REPO / 'analysis' / 'notebooks' / 'notebook_outputs'

TARIFF_SOURCE = RAW / 'tariff_hs_codes_8_27_2026.csv'
CONCORDANCE = RAW / 'C616_HS8toNaics6_concord_202505.csv'
EXPORT_DIR = RAW / 'exports'
ESTABLISHMENTS = REPO / 'analysis' / 'input-data' / 'large_size_data' / 'Dec2022_Estabcounts_byDA.csv'
ADA_FULL = INTERMEDIATE / 'trail_ada_full.csv'
CSD_FULL = INTERMEDIATE / 'trail_csd_full.csv'
ADA_HOME = NOTEBOOK_OUTPUTS / 'trail7.csv'
CSD_HOME = NOTEBOOK_OUTPUTS / 'trail7_csd.csv'
DA_TO_ADA = INTERMEDIATE / 'ada_da_relation.csv'
DA_SHAPEFILE = REPO / 'analysis' / 'input-data' / 'large_size_data' / 'lda_000b21a_e.shp'
CSD_SHAPEFILE = REPO / 'data' / 'census' / 'lcsd000b21a_e'

ADA_CENTROIDS = NOTEBOOK_OUTPUTS / 'centroids.geojson'
ADA_CHOROPLETH = NOTEBOOK_OUTPUTS / 'choropleth.geojson'
CSD_CENTROIDS = NOTEBOOK_OUTPUTS / 'centroids_csd.geojson'
CSD_CHOROPLETH = NOTEBOOK_OUTPUTS / 'choropleth_csd.geojson'

CATEGORIES = {
    'before August 22': 'BeforeAug22',
    'after August 22': 'AfterAug22',
    'Section 338': 'Section338',
}
PROVINCES = ['NL', 'PEI', 'NS', 'NB', 'QC', 'ON', 'MB', 'SK', 'AL', 'BC', 'YK', 'NWT', 'NU']
WRITE_OUTPUTS = True

print('Repository:', REPO)
print('Categories:', CATEGORIES)
print('DA shapefile exists:', DA_SHAPEFILE.exists())
print('CSD shapefile exists:', CSD_SHAPEFILE.exists())

Repository: d:\coding\SoC\tariffs\tariffs
Categories: {'before August 22': 'BeforeAug22', 'after August 22': 'AfterAug22', 'Section 338': 'Section338'}
DA shapefile exists: True
CSD shapefile exists: True


In [26]:
# Load the three new HS-code groups and normalize to HS6.
tariffs = pd.read_csv(TARIFF_SOURCE, dtype={'HS Code': str})
tariffs['HS_Code_6digit'] = (
    tariffs['HS Code'].astype(str).str.replace(r'\D', '', regex=True).str[:6]
)
tariffs = tariffs[tariffs['Category'].isin(CATEGORIES)].copy()
tariffs = tariffs[['HS_Code_6digit', 'Category']].drop_duplicates()
print(tariffs.groupby('Category').size().to_string())
print('Unique HS6:', tariffs['HS_Code_6digit'].nunique())

Category
Section 338          371
after August 22     1108
before August 22     755
Unique HS6: 1108


In [27]:
# Build province-specific NAICS export rates using the same weighting rule as the main notebooks.
concordance = pd.read_csv(CONCORDANCE, dtype=str)
concordance['HS_Code_6digit'] = (
    concordance['HS8 Code'].astype(str).str.replace(r'\D', '', regex=True).str.zfill(8).str[:6]
)
concordance['NAICS'] = concordance['NAICS 6 Code'].astype(str).str.replace(r'\D', '', regex=True).str[:6]
concordance = concordance[['HS_Code_6digit', 'NAICS']].drop_duplicates()
naics_tariffs = tariffs.merge(concordance, on='HS_Code_6digit', how='inner')
naics_tariffs['count'] = naics_tariffs.groupby(['HS_Code_6digit', 'Category'])['NAICS'].transform('count')

rate_rows = []
for province in PROVINCES:
    global_df = pd.read_csv(EXPORT_DIR / f'{province}-Global.csv', usecols=['Commodity', 'Value ($)'])
    global_df['HS_Code_6digit'] = global_df['Commodity'].astype(str).str.replace(r'\D', '', regex=True).str[:6]
    global_df = global_df.groupby('HS_Code_6digit', as_index=False)['Value ($)'].sum().rename(columns={'Value ($)': 'global_value'})
    us_df = pd.read_csv(EXPORT_DIR / f'{province}-US.csv', usecols=['Commodity', 'Value ($)'])
    us_df['HS_Code_6digit'] = us_df['Commodity'].astype(str).str.replace(r'\D', '', regex=True).str[:6]
    us_df = us_df.groupby('HS_Code_6digit', as_index=False)['Value ($)'].sum().rename(columns={'Value ($)': 'us_value'})
    joined = naics_tariffs.merge(global_df, on='HS_Code_6digit', how='left').merge(us_df, on='HS_Code_6digit', how='left')
    joined[['global_value', 'us_value']] = joined[['global_value', 'us_value']].fillna(0)
    joined['rate_value'] = (joined['us_value'] / joined['count']) / (joined['global_value'] / joined['count'])
    joined['Province'] = province
    rate_rows.append(joined[['Category', 'NAICS', 'Province', 'rate_value']])

rates = pd.concat(rate_rows, ignore_index=True)
rates['rate_value'] = rates['rate_value'].replace([np.inf, -np.inf], np.nan).fillna(0)
rates = rates.groupby(['Category', 'NAICS', 'Province'], as_index=False)['rate_value'].sum()
print('Mapped HS-NAICS rows:', len(naics_tariffs))
print(rates.groupby('Category')['NAICS'].nunique().to_string())

Mapped HS-NAICS rows: 2493
Category
Section 338         134
after August 22     208
before August 22    120


In [28]:
def province_from_dauid(dauid_str):
    s = str(dauid_str).strip()
    if len(s) < 2:
        return None
    code = s[:2]
    mapping = {
        '10':'NL','11':'PEI','12':'NS','13':'NB',
        '24':'QC','35':'ON','46':'MB','47':'SK',
        '48':'AL','59':'BC','60':'YK','61':'NWT','62':'NU'
    }
    return mapping.get(code)

In [29]:
# Calculate new workplace counts from the retained ADA employment table.
def province_from_id(value):
    code = str(value)[9:11]
    return {'10':'NL','11':'PEI','12':'NS','13':'NB','24':'QC','35':'ON','46':'MB','47':'SK','48':'AL','59':'BC','60':'YK','61':'NWT','62':'NU'}.get(code)

def category_work(full_path, id_col):
    full = pd.read_csv(full_path, dtype={id_col: str})
    full['Province'] = full[id_col].map(province_from_id)
    result = full[[id_col]].copy()
    for category, prefix in CATEGORIES.items():
        category_rates = rates[rates['Category'] == category]
        values = pd.Series(0.0, index=full.index)
        for province in PROVINCES:
            province_rates = category_rates[category_rates['Province'] == province].set_index('NAICS')['rate_value']
            columns = [c for c in province_rates.index if c in full.columns]
            if columns:
                values.loc[full['Province'] == province] = np.ceil(full.loc[full['Province'] == province, columns].mul(province_rates[columns].to_numpy(), axis=1).sum(axis=1))
        result[f'{prefix}_E'] = values.astype('int64')
    return result

ada_work = category_work(ADA_FULL, 'ADADGUID')
csd_work = category_work(CSD_FULL, 'CSDDGUID')
print('ADA workplace fields:', ada_work.iloc[:, 1:].sum().to_dict())
print('CSD workplace fields:', csd_work.iloc[:, 1:].sum().to_dict())

ADA workplace fields: {'BeforeAug22_E': 3960086, 'AfterAug22_E': 6234216, 'Section338_E': 2491394}
CSD workplace fields: {'BeforeAug22_E': 3959437, 'AfterAug22_E': 6233559, 'Section338_E': 2490749}


In [30]:
# Calculate residence counts from the retained, export-weighted NAICS residence tables.
def category_home(home_path, id_col):
    home = pd.read_csv(home_path, dtype={id_col: str})
    result = home[[id_col]].copy()
    naics_columns = [c for c in home.columns if c not in {id_col, 'Unnamed: 0', 'Province', 'To', 'Sum'}]
    for category, prefix in CATEGORIES.items():
        category_naics = set(rates.loc[rates['Category'] == category, 'NAICS'])
        columns = [c for c in naics_columns if c in category_naics]
        result[f'{prefix}_C'] = home[columns].sum(axis=1) if columns else 0
    return result

ada_home = category_home(ADA_HOME, 'ADADGUID')
csd_home = category_home(CSD_HOME, 'CSDDGUID')
print('ADA residence fields:', ada_home.iloc[:, 1:].sum().to_dict())
print('CSD residence fields:', csd_home.iloc[:, 1:].sum().to_dict())

ADA residence fields: {'BeforeAug22_C': 1169749.0, 'AfterAug22_C': 1823435.0, 'Section338_C': 1205081.0}
CSD residence fields: {'BeforeAug22_C': 1022128.0, 'AfterAug22_C': 1582241.0, 'Section338_C': 1046685.0}


In [31]:
# Merge the new fields into the existing ADA/CSD centroid and choropleth files.
def add_fields(path, id_col, counts, work, home, output_path=None):
    gdf = gpd.read_file(path)
    for frame in [counts, work, home]:
        gdf = gdf.merge(frame, on=id_col, how='left', validate='one_to_one')
    count_prefixes = list(CATEGORIES.values())
    for prefix in count_prefixes:
        for suffix in ['_B', '_E', '_C']:
            column = f'{prefix}{suffix}'
            if column not in gdf.columns:
                gdf[column] = 0
    if output_path is not None:
        gdf.to_file(output_path, driver='GeoJSON')
    return gdf

# Business counts require the establishment table. This cell is intentionally separate because it is the only large scan.
print('The next cell calculates business counts and may take several minutes.')

The next cell calculates business counts and may take several minutes.


In [32]:
sample = pd.read_csv(ESTABLISHMENTS, encoding='ISO-8859-1', nrows=10,
                     dtype={'DisseminationAre': str})   # force string to preserve leading zeros
print(sample[['DisseminationAre', 'NAICS']].head())

  DisseminationAre                                      NAICS
0         10000000                                      Total
1         10000000                               Unclassified
2         10000000                      Sub-total, classified
3         10000000                   111110 - Soybean farming
4         10000000  111120 - Oilseed (except soybean) farming


In [33]:
# Compute business counts by DA from establishments
all_cols = pd.read_csv(ESTABLISHMENTS, encoding='ISO-8859-1', nrows=1).columns.tolist()
business_cols = [c for c in ['DisseminationAre', 'NAICS', 'Total, with employees'] if c in all_cols]
business_chunks = []
rate_wide = rates.pivot_table(index=['NAICS', 'Province'], columns='Category', values='rate_value', aggfunc='sum', fill_value=0).reset_index()

for chunk in pd.read_csv(ESTABLISHMENTS, encoding='ISO-8859-1', usecols=business_cols, chunksize=1_000_000):
    chunk = chunk.rename(columns={'DisseminationAre': 'DAUID', 'Total, with employees': 'businesses'})
    chunk['DAUID'] = (
        chunk['DAUID'].astype(str).str.strip()
        .str.replace(r'\.0$', '', regex=True)
        .str.zfill(8)
    )
    chunk['NAICS'] = chunk['NAICS'].astype(str).str.strip().str[:6]
    chunk = chunk.dropna(subset=['NAICS'])
    chunk['Province'] = chunk['DAUID'].apply(province_from_dauid)
    chunk = chunk.merge(rate_wide, on=['NAICS', 'Province'], how='inner')
    for category, prefix in CATEGORIES.items():
        chunk[f'{prefix}_B'] = np.ceil(chunk['businesses'].fillna(0) * chunk[category])
    business_chunks.append(chunk.groupby('DAUID')[[f'{prefix}_B' for prefix in CATEGORIES.values()]].sum())

ada_business_by_da = pd.concat(business_chunks).groupby(level=0).sum().reset_index()
print('ada_business_by_da DAUID dtype:', ada_business_by_da['DAUID'].dtype)

# Load DA shapefile to get mapping DAUID → DADGUID
da_shape = gpd.read_file(DA_SHAPEFILE)[['DAUID', 'DGUID']].copy()
da_shape = da_shape.rename(columns={'DGUID': 'DADGUID'})
da_shape['DAUID'] = da_shape['DAUID'].astype(str).str.strip().str.zfill(8)

# Now map DAUID to DADGUID and then to ADADGUID
da_to_ada = pd.read_csv(DA_TO_ADA, dtype=str)
ada_business = (
    ada_business_by_da
    .merge(da_shape, on='DAUID', how='inner')
    .merge(da_to_ada, on='DADGUID', how='inner')
    .groupby('ADADGUID', as_index=False)[[f'{prefix}_B' for prefix in CATEGORIES.values()]].sum()
)

# For CSD: reuse da_csd (from the DA/CSD spatial join) to get CSD business counts
da = gpd.read_file(DA_SHAPEFILE)[['DAUID', 'DGUID', 'geometry']].copy()
da = da.rename(columns={'DGUID': 'DADGUID'})
csd = gpd.read_file(CSD_SHAPEFILE)[['DGUID', 'geometry']].copy()
csd = csd.rename(columns={'DGUID': 'CSDDGUID'})
da = da.to_crs('EPSG:3347')
csd = csd.to_crs('EPSG:3347')
da_points = da.copy()
da_points['geometry'] = da_points.geometry.representative_point()
da_csd = gpd.sjoin(da_points, csd, how='left', predicate='within').drop(columns=['index_right', 'geometry'])
da_csd['DAUID'] = da_csd['DAUID'].astype(str).str.strip().str.zfill(8)  # was pd.to_numeric(...).astype('Int64') -- broke the merge below

csd_business = (
    da_csd
    .merge(ada_business_by_da, on='DAUID', how='inner')
    .groupby('CSDDGUID', as_index=False)[[f'{prefix}_B' for prefix in CATEGORIES.values()]].sum()
)

print('ADA business rows:', len(ada_business), 'CSD business rows:', len(csd_business))

ada_business_by_da DAUID dtype: object
ADA business rows: 4757 CSD business rows: 4184


In [34]:
# Build new fields and validate before writing.
ada_new = ada_business.merge(ada_work, on='ADADGUID', how='outer').merge(ada_home, on='ADADGUID', how='outer').fillna(0)
csd_new = csd_business.merge(csd_work, on='CSDDGUID', how='outer').merge(csd_home, on='CSDDGUID', how='outer').fillna(0)
for frame in [ada_new, csd_new]:
    for column in frame.columns:
        if column not in ('ADADGUID', 'CSDDGUID'):
            frame[column] = frame[column].astype('int64')

print('ADA new totals:', ada_new.drop(columns='ADADGUID').sum().to_dict())
print('CSD new totals:', csd_new.drop(columns='CSDDGUID').sum().to_dict())
print('WRITE_OUTPUTS =', WRITE_OUTPUTS)

ADA new totals: {'BeforeAug22_B': 111648, 'AfterAug22_B': 216026, 'Section338_B': 114472, 'BeforeAug22_E': 3960086, 'AfterAug22_E': 6234216, 'Section338_E': 2491394, 'BeforeAug22_C': 1169749, 'AfterAug22_C': 1823435, 'Section338_C': 1205081}
CSD new totals: {'BeforeAug22_B': 111648, 'AfterAug22_B': 216026, 'Section338_B': 114472, 'BeforeAug22_E': 3959437, 'AfterAug22_E': 6233559, 'Section338_E': 2490749, 'BeforeAug22_C': 1022128, 'AfterAug22_C': 1582241, 'Section338_C': 1046685}
WRITE_OUTPUTS = True


In [35]:
# Append to CSV and GeoJSON outputs only after reviewing the validation above.
def merge_csv(path, id_col, new_fields):
    old = pd.read_csv(path, dtype={id_col: str})
    old = old.drop(columns=[c for c in new_fields.columns if c != id_col and c in old.columns])
    merged = old.merge(new_fields, on=id_col, how='left', validate='one_to_one').fillna(0)
    merged.to_csv(path, index=False)

def merge_geojson(path, id_col, new_fields):
    gdf = gpd.read_file(path)
    gdf = gdf.drop(columns=[c for c in new_fields.columns if c != id_col and c in gdf.columns])
    gdf = gdf.merge(new_fields, on=id_col, how='left', validate='one_to_one').fillna(0)
    gdf.to_file(path, driver='GeoJSON')

if WRITE_OUTPUTS:
    merge_geojson(ADA_CENTROIDS, 'ADADGUID', ada_new)
    merge_geojson(ADA_CHOROPLETH, 'ADADGUID', ada_new[['ADADGUID'] + [f'{p}_B' for p in CATEGORIES.values()] + [f'{p}_E' for p in CATEGORIES.values()] + [f'{p}_C' for p in CATEGORIES.values()]])
    merge_geojson(CSD_CENTROIDS, 'CSDDGUID', csd_new)
    merge_geojson(CSD_CHOROPLETH, 'CSDDGUID', csd_new[['CSDDGUID'] + [f'{p}_B' for p in CATEGORIES.values()] + [f'{p}_E' for p in CATEGORIES.values()] + [f'{p}_C' for p in CATEGORIES.values()]])
    print('GeoJSON outputs updated.')
else:
    print('Dry run only. Set WRITE_OUTPUTS = True and rerun this cell to write outputs.')

KeyboardInterrupt: 

In [ ]:
business_chunks = []
rate_wide = rates.pivot_table(index=['NAICS', 'Province'], columns='Category', values='rate_value',
                              aggfunc='sum', fill_value=0).reset_index()

print("rate_wide shape:", rate_wide.shape)
print("rate_wide head:\n", rate_wide.head())

for i, chunk in enumerate(pd.read_csv(ESTABLISHMENTS, encoding='ISO-8859-1', usecols=business_cols, chunksize=1_000_000)):
    print(f"\n--- Chunk {i+1} ---")
    print("Original chunk rows:", len(chunk))
    
    chunk = chunk.rename(columns={'DisseminationAre': 'DAUID', 'Total, with employees': 'businesses'})
    chunk['NAICS'] = chunk['NAICS'].astype(str).str[:6]
    chunk['Province'] = chunk['DAUID'].apply(province_from_dauid)
    
    print("After NAICS & Province:", len(chunk))
    print("Sample NAICS:", chunk['NAICS'].head(3).tolist())
    print("Sample Province:", chunk['Province'].head(3).tolist())
    print("Unique Provinces:", chunk['Province'].unique())
    
    # Drop rows without province (should be few)
    before_merge = len(chunk)
    chunk = chunk.dropna(subset=['Province'])
    print("After dropping NA province:", len(chunk))
    
    # Merge with rate_wide
    chunk_merged = chunk.merge(rate_wide, on=['NAICS', 'Province'], how='inner')
    print("After merge with rate_wide:", len(chunk_merged))
    
    if len(chunk_merged) == 0:
        # Show some NAICS values from chunk and rate_wide to compare
        print("NAICS from chunk (first 5):", chunk['NAICS'].head(5).tolist())
        print("NAICS from rate_wide (first 5):", rate_wide['NAICS'].head(5).tolist())
        print("Province from chunk (first 5):", chunk['Province'].head(5).tolist())
        print("Province from rate_wide (first 5):", rate_wide['Province'].head(5).tolist())
        # Possibly break to avoid many prints
        break
    
    # Continue with business calculations
    for category, prefix in CATEGORIES.items():
        chunk_merged[f'{prefix}_B'] = np.ceil(chunk_merged['businesses'].fillna(0) * chunk_merged[category])
    
    grouped = chunk_merged.groupby('DAUID')[[f'{prefix}_B' for prefix in CATEGORIES.values()]].sum()
    print("Grouped rows:", len(grouped))
    business_chunks.append(grouped)

rate_wide shape: (2704, 5)
rate_wide head:
 Category   NAICS Province  Section 338  after August 22  before August 22
0         111219       AL     0.951784         0.951784               0.0
1         111219       BC     1.000000         1.000000               0.0
2         111219       MB     1.000000         1.000000               0.0
3         111219       NB     1.000000         1.000000               0.0
4         111219       NL     0.000000         0.000000               0.0

--- Chunk 1 ---
Original chunk rows: 1000000
After NAICS & Province: 1000000
Sample NAICS: ['Total', 'Unclas', 'Sub-to']
Sample Province: ['NL', 'NL', 'NL']
Unique Provinces: ['NL' 'PEI']
After dropping NA province: 1000000
After merge with rate_wide: 224640
Grouped rows: 1080

--- Chunk 2 ---
Original chunk rows: 1000000
After NAICS & Province: 1000000
Sample NAICS: ['721192', '721198', '721211']
Sample Province: ['PEI', 'PEI', 'PEI']
Unique Provinces: ['PEI' 'NS']
After dropping NA province: 1000000
Afte

KeyboardInterrupt: 